# Post Request in FastAPI

![post](post_request_flow.png)

# Post Request in FastAPI

---

## Setup

```python
from fastapi import FastAPI, Path, HTTPException, Query
import json

# Initialize the FastAPI app
app = FastAPI()

# Helper — load patient data from JSON file
def load_file():
    with open("patient.json", "r") as f:
        data = json.load(f)
    return data
```

---

## 1) Basic Pydantic Model for Request Body Validation

When a client sends data in the request body, FastAPI uses a **Pydantic model** to validate it automatically.

> Install Pydantic: `pip install pydantic`

We inherit from `BaseModel` and define the expected fields and their types:

```python
from fastapi import FastAPI, Path, HTTPException, Query
from pydantic import BaseModel
import json

class Patient(BaseModel):
    id:     str
    name:   str
    gender: str
    age:    int
    city:   str
    height: float
    weight: float

app = FastAPI()

def load_file():
    with open("patient.json", "r") as f:
        data = json.load(f)
    return data
```

---

## 2) Adding Field Constraints & Descriptions with `Annotated`

The basic model above does generic type validation only. To add **rules, descriptions, and examples** to each field, use `Annotated` combined with `Field`.

**Syntax:**
```python
field_name: Annotated[datatype, Field(..., description='', example='', gt=0, ...)]
```

Use `Literal` from `typing` to restrict a field to a fixed set of allowed values (e.g. `'male'`, `'female'`, `'others'`).

```python
from fastapi import FastAPI, Path, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Annotated, Literal
import json

class Patient(BaseModel):
    id:     Annotated[str,   Field(..., description='Unique ID of the patient',    example='P001')]
    name:   Annotated[str,   Field(..., description='Full name of the patient')]
    gender: Annotated[Literal['male', 'female', 'others'], Field(..., description='Gender of the patient')]
    age:    Annotated[int,   Field(..., description='Age of the patient',           gt=0, lt=101)]
    city:   Annotated[str,   Field(..., description='City of the patient')]
    height: Annotated[float, Field(..., description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., description='Weight of the patient in kg')]

app = FastAPI()

def load_file():
    with open("patient.json", "r") as f:
        data = json.load(f)
    return data
```

---

## 3) Computed Fields — Deriving BMI & Health Verdict

Use `@computed_field` to automatically calculate a new field from existing ones. No need to pass these in the request — Pydantic computes them.

**Syntax:**
```python
@computed_field
@property
def function_name(self) -> return_datatype:
    # calculation
    return result
```

```python
from fastapi import FastAPI, Path, HTTPException, Query
from pydantic import BaseModel, Field, computed_field
from typing import Annotated, Literal
import json

class Patient(BaseModel):
    id:     Annotated[str,   Field(..., description='Unique ID of the patient',    example='P001')]
    name:   Annotated[str,   Field(..., description='Full name of the patient')]
    gender: Annotated[Literal['male', 'female', 'others'], Field(..., description='Gender of the patient')]
    age:    Annotated[int,   Field(..., description='Age of the patient',           gt=0, lt=101)]
    city:   Annotated[str,   Field(..., description='City of the patient')]
    height: Annotated[float, Field(..., description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., description='Weight of the patient in kg')]

    # Computed field 1 — BMI
    @computed_field
    @property
    def bmi(self) -> float:
        # self.weight and self.height refer to the fields defined above
        return round(self.weight / (self.height ** 2), 2)

    # Computed field 2 — Health verdict based on BMI
    @computed_field
    @property
    def verdict(self) -> str:
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 25:
            return 'Normal'
        else:
            return 'Obese'

app = FastAPI()

def load_file():
    with open("patient.json", "r") as f:
        data = json.load(f)
    return data
```

---

## 4) POST Endpoint — Create a New Patient

This endpoint will:
1. Load existing patient data from the JSON file
2. Check whether the patient already exists
3. Add the new patient to the data and save it
4. Return a JSON response

```python
from fastapi import FastAPI, Path, HTTPException, Query
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, computed_field
from typing import Annotated, Literal
import json

# ── Pydantic Model ────────────────────────────────────────────────────────────

class Patient(BaseModel):
    id:     Annotated[str,   Field(..., description='Unique ID of the patient',    example='P001')]
    name:   Annotated[str,   Field(..., description='Full name of the patient')]
    gender: Annotated[Literal['male', 'female', 'others'], Field(..., description='Gender of the patient')]
    age:    Annotated[int,   Field(..., description='Age of the patient',           gt=0, lt=101)]
    city:   Annotated[str,   Field(..., description='City of the patient')]
    height: Annotated[float, Field(..., description='Height of the patient in metres')]
    weight: Annotated[float, Field(..., description='Weight of the patient in kg')]

    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / (self.height ** 2), 2)

    @computed_field
    @property
    def verdict(self) -> str:
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 25:
            return 'Normal'
        else:
            return 'Obese'

# ── App & Helpers ─────────────────────────────────────────────────────────────

app = FastAPI()

def load_data():
    with open('patient.json', 'r') as f:
        return json.load(f)

def save_data(data):
    with open('patient.json', 'w') as f:
        json.dump(data, f)  # json.dump writes a Python object into a file as JSON

# ── Endpoint ──────────────────────────────────────────────────────────────────

@app.post('/create')
def create_patient(patient: Patient):

    # Step 1 — Load existing data
    data = load_data()

    # Step 2 — Check if patient already exists
    if patient.id in data:
        raise HTTPException(status_code=400, detail='Patient already exists')

    # Step 3 — Add new patient (excluding 'id' since it's already used as the key)
    data[patient.id] = patient.model_dump(exclude={'id'})

    # Step 4 — Save updated data back to file
    save_data(data)

    # Step 5 — Return success response
    return JSONResponse(status_code=201, content={'message': 'Patient created successfully'})
```

> **Note:** `model_dump(exclude={'id'})` converts the Pydantic object to a dictionary while leaving out the `id` field — because `id` is already being used as the dictionary key, so storing it again inside the value would be redundant.

---

## Quick Reference

| Concept | What it does |
|---|---|
| `BaseModel` | Defines the schema and enables automatic type validation |
| `Field(...)` | Adds constraints, descriptions, and examples to a field |
| `Annotated` | Attaches `Field` metadata to a type hint |
| `Literal` | Restricts a field to specific allowed values |
| `@computed_field` | Derives a new field from existing fields automatically |
| `model_dump()` | Converts a Pydantic object → Python dictionary |
| `model_dump(exclude={...})` | Same, but omits specified fields |
| `HTTPException` | Returns an error response with a status code and message |
| `JSONResponse` | Returns a custom JSON response with a specific status code |

## You can the working by going to `/docs` and there u will see the post endpoint by the name of `/create` and then by clicking on it u will see the request body just send the data you will see that `New Data will be added on the Json`